In [ ]:
from pathlib import Path
Path.cwd()

In [ ]:
# GTFS Realtime API Call (Environment-secured)

import requests
from dotenv import load_dotenv
from pathlib import Path
import os

# Load environment variables
PROJECT_ROOT = Path.cwd().parent  # adjust if needed
load_dotenv(PROJECT_ROOT / ".env")

# Get API key securely
API_KEY = os.getenv("AT_API_KEY")

if not API_KEY:
    raise ValueError("AT_API_KEY not found. Add it to your .env file.")

# Endpoint
url = "https://api.at.govt.nz/realtime/legacy/tripupdates"

# Headers
headers = {
    "Ocp-Apim-Subscription-Key": API_KEY
}

# Request
response = requests.get(url, headers=headers)

# Output check
print("Status Code:", response.status_code)
print("Content Length:", len(response.content))

In [ ]:
from datetime import datetime
import os

# Create folder if not exists
save_dir = "data/raw/gtfs_realtime"
os.makedirs(save_dir, exist_ok=True)

# Timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# File path
file_path = f"{save_dir}/tripupdates_{timestamp}.pb"

# Save file
with open(file_path, "wb") as f:
    f.write(response.content)

print("Saved file:", file_path)

In [ ]:
print("Content-Type:", response.headers.get("Content-Type"))
print("First 200 bytes:")
print(response.content[:200])

In [ ]:
from datetime import datetime
import json
from pathlib import Path

save_dir = Path("data/raw/gtfs_realtime")
save_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
json_path = save_dir / f"tripupdates_{timestamp}.json"

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(response.json(), f, indent=2)

print("Saved JSON file:", json_path)

In [ ]:
#load JSON 
import json

file_path = file_path = "../data/raw/gtfs_realtime/tripupdates_20260430_204204.json"

with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Keys:", data.keys())
print("Entity count:", len(data["response"]["entity"]))

In [ ]:
from pathlib import Path
Path.cwd()

In [ ]:
#Extract Data (Core Logic)
import pandas as pd
records = []

entities = data["response"]["entity"]

for entity in entities:
    trip_update = entity.get("trip_update", {})
    trip = trip_update.get("trip", {})

    records.append({
        "entity_id": entity.get("id"),
        "trip_id": trip.get("trip_id"),
        "route_id": trip.get("route_id"),
        "direction_id": trip.get("direction_id"),
        "start_time": trip.get("start_time"),
        "start_date": trip.get("start_date"),
        "timestamp": trip_update.get("timestamp"),
        "delay_seconds": trip_update.get("delay"),
        "is_deleted": entity.get("is_deleted")
    })

df_realtime = pd.DataFrame(records)

print("Realtime records:", df_realtime.shape)
df_realtime.head()

In [ ]:
df_realtime["delay_minutes"] = df_realtime["delay_seconds"] / 60

df_realtime[["trip_id", "route_id", "timestamp", "delay_seconds", "delay_minutes"]].head()

In [ ]:
#Loading GTFS Static (routes)
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
GTFS_STATIC_PATH = PROJECT_ROOT / "data" / "raw" / "gtfs_static"

routes = pd.read_csv(GTFS_STATIC_PATH / "routes.txt")

print("Project root:", PROJECT_ROOT)
print("Routes shape:", routes.shape)
routes.head()

In [ ]:
#Joining Realtime + Static Data#
df_realtime_routes = df_realtime.merge(
    routes[["route_id", "route_short_name", "route_long_name", "route_type"]],
    on="route_id",
    how="left"
)

print("Realtime + Routes shape:", df_realtime_routes.shape)

df_realtime_routes[
    ["trip_id", "route_id", "route_short_name", "route_long_name", "delay_seconds", "route_type", "delay_minutes"]
].head()

In [ ]:
PROJECT_ROOT = Path.cwd().parent

In [ ]:
#check quality of data integration
issing_routes = df_realtime_routes["route_short_name"].isna().sum()

print("Missing route matches:", missing_routes)
print("Match rate:", round((1 - missing_routes / len(df_realtime_routes)) * 100, 2), "%")

In [ ]:
#Calculate Average Delay per Route
route_delay = df_realtime_routes.groupby("route_short_name")["delay_minutes"].mean().sort_values(ascending=False)

route_delay.head(10)

In [ ]:
#Most delayed routes visualization
import matplotlib.pyplot as plt

top_delay = route_delay.head(10)

top_delay.plot(kind="bar", figsize=(10,5))

plt.title("Top 10 Routes by Average Delay (Minutes)")
plt.ylabel("Average Delay (Minutes)")
plt.xlabel("Route")
plt.xticks(rotation=45)

plt.show()

In [ ]:
#Most Consistently Late Routes
late_ratio = df_realtime_routes.groupby("route_short_name")["delay_minutes"].apply(lambda x: (x > 0).mean())

late_ratio = late_ratio.sort_values(ascending=False)

late_ratio.head(10)

In [ ]:
df_realtime_routes["route_type"].value_counts()

In [ ]:
def map_transport(x):
    if x == 3:
        return "Bus"
    elif x == 2:
        return "Train"
    elif x == 4:
        return "Ferry"
    else:
        return "Other"

df_realtime_routes["transport_mode"] = df_realtime_routes["route_type"].apply(map_transport)

df_realtime_routes["transport_mode"].value_counts()

In [ ]:
df_realtime_routes.groupby("transport_mode")["delay_minutes"].mean()

In [ ]:
#add median delay for more robust analysis
df_realtime_routes.groupby("transport_mode")["delay_minutes"].median()

In [ ]:
#delay distribution visual
import matplotlib.pyplot as plt

df_realtime_routes.boxplot(column="delay_minutes", by="transport_mode", figsize=(8,5))

plt.title("Delay Distribution by Transport Mode")
plt.suptitle("")
plt.ylabel("Delay (Minutes)")
plt.show()

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
INTERIM_PATH = PROJECT_ROOT / "data" / "interim"

INTERIM_PATH.exists()

In [ ]:
#save integrated data set
output_path = INTERIM_PATH / "realtime_with_routes_sample.csv"

df_realtime_routes.to_csv(output_path, index=False)

print("Saved to:", output_path)
